# 00c — Obtain and prepare the Ghostbuster source text

This notebook starts from the public Ghostbuster data repository and builds the raw-text tables used by our \(U(k)\), \(R(d)\), and \(F\) extractors.

Pinned data revision:

```text
vivek3141/ghostbuster-data
f73eecbbad1694f02ecb1c98040af8da206f0f49
```

Outputs:

- `data/raw/ghostbuster_essays.parquet`
- `data/raw/ghostbuster_detector_pool.parquet`
- `data/splits/ghostbuster_detector_manifest.parquet`

Ghostbuster stores:
- `essay/` and `wp/` documents directly under each generator directory;
- `reuter/` documents inside nested author directories.

The helper below therefore uses flat discovery for essay/WritingPrompts and recursive discovery only for Reuters. This avoids accidentally treating metadata such as `logprobs/*.txt` as documents.

In [1]:
from pathlib import Path
import subprocess
import hashlib
import pandas as pd
import numpy as np

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "experiments" else cwd

DATA = ROOT / "data"
RAW = DATA / "raw"
SPLITS = DATA / "splits"
EXTERNAL = DATA / "external"

RAW.mkdir(parents=True, exist_ok=True)
SPLITS.mkdir(parents=True, exist_ok=True)
EXTERNAL.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/vivek3141/ghostbuster-data.git"
DATA_COMMIT = "f73eecbbad1694f02ecb1c98040af8da206f0f49"
GB_DATA = EXTERNAL / "ghostbuster-data"

print("ROOT:", ROOT)
print("Ghostbuster checkout:", GB_DATA)

ROOT: ..
Ghostbuster checkout: ../data/external/ghostbuster-data


## Clone and pin the public data

In [2]:
if not (GB_DATA / ".git").exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(GB_DATA)],
        check=True,
    )

subprocess.run(
    ["git", "-C", str(GB_DATA), "fetch", "--all", "--tags"],
    check=True,
)

subprocess.run(
    ["git", "-C", str(GB_DATA), "checkout", DATA_COMMIT],
    check=True,
)

actual_commit = subprocess.check_output(
    ["git", "-C", str(GB_DATA), "rev-parse", "HEAD"],
    text=True,
).strip()

assert actual_commit == DATA_COMMIT
print("Pinned commit:", actual_commit)

Pinned commit: f73eecbbad1694f02ecb1c98040af8da206f0f49


HEAD is now at f73eecbba Fixing Essay Data


## Helpers

The crucial rule is:

- `essay`, `wp`: use only top-level `.txt` files;
- `reuter`: recurse through author directories, while excluding metadata directories.

This prevents files such as `logprobs/1-ada.txt` from entering the document table.

In [3]:
EXCLUDED_DIR_NAMES = {
    "logprobs",
    "prompts",
    "__pycache__",
}


def read_text(path: Path):
    return path.read_text(
        encoding="utf-8",
        errors="ignore",
    ).strip()


def text_files(generator_root: Path, domain: str):
    assert generator_root.exists(), f"Missing directory: {generator_root}"

    if domain in {"essay", "wp"}:
        files = [
            p for p in generator_root.glob("*.txt")
            if p.is_file()
        ]
    elif domain == "reuter":
        files = []
        for p in generator_root.rglob("*.txt"):
            if not p.is_file():
                continue

            rel_parts = {
                part.lower()
                for part in p.relative_to(generator_root).parts[:-1]
            }

            if rel_parts & EXCLUDED_DIR_NAMES:
                continue

            files.append(p)
    else:
        raise ValueError(f"Unknown Ghostbuster domain: {domain}")

    return sorted(
        files,
        key=lambda p: p.relative_to(generator_root).as_posix(),
    )


def document_id(path: Path, generator_root: Path, domain: str):
    rel = path.relative_to(generator_root)

    if domain in {"essay", "wp"}:
        if not path.stem.isdigit():
            raise ValueError(
                f"Expected numeric filename for {domain}: {rel}"
            )
        return int(path.stem)

    # Reuters: preserve nested author/article path to avoid collisions.
    return rel.with_suffix("").as_posix()


def rel_key(path: Path):
    return path.relative_to(GB_DATA).as_posix()


def sha256_text(text: str):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

## Build the matched essay corpus

Seven essay sources are retained:

- human
- gpt
- claude
- gpt_prompt1
- gpt_prompt2
- gpt_semantic
- gpt_writing

Only IDs present in all seven directories are kept.

In [4]:
ESSAY_GENERATORS = [
    "human",
    "gpt",
    "claude",
    "gpt_prompt1",
    "gpt_prompt2",
    "gpt_semantic",
    "gpt_writing",
]

essay_files = {}

for generator in ESSAY_GENERATORS:
    root = GB_DATA / "essay" / generator
    files = text_files(root, domain="essay")

    essay_files[generator] = {
        str(document_id(p, root, "essay")): p
        for p in files
    }

    print(generator, "essay docs:", len(files))

common_ids = set.intersection(
    *[
        set(mapping.keys())
        for mapping in essay_files.values()
    ]
)

common_ids = sorted(common_ids, key=int)

rows = []

for gid in common_ids:
    for generator in ESSAY_GENERATORS:
        p = essay_files[generator][gid]
        text = read_text(p)

        rows.append({
            "row_id": f"essay::{generator}::{gid}",
            "file": rel_key(p),
            "domain": "essay",
            "generator": generator,
            "label": int(generator != "human"),
            "group_id": int(gid),
            "text": text,
            "text_sha256": sha256_text(text),
        })

essays = pd.DataFrame(rows)
# Normalize identifier type for portable parquet serialization.
essays["group_id"] = essays["group_id"].astype(str)

assert not essays["row_id"].duplicated().any()
assert essays.groupby("generator").size().nunique() == 1
assert (
    essays.groupby("group_id")["generator"]
    .nunique()
    .eq(len(ESSAY_GENERATORS))
    .all()
)

ESSAY_OUT = RAW / "ghostbuster_essays.parquet"
essays.to_parquet(ESSAY_OUT, index=False)

print("saved:", ESSAY_OUT)
print("shape:", essays.shape)
display(essays.groupby("generator").size().to_frame("n"))

human essay docs: 1000
gpt essay docs: 1000
claude essay docs: 1000
gpt_prompt1 essay docs: 1000
gpt_prompt2 essay docs: 1000
gpt_semantic essay docs: 1000
gpt_writing essay docs: 1000
saved: ../data/raw/ghostbuster_essays.parquet
shape: (7000, 8)


,n
generator,
claude,1000
gpt,1000
gpt_prompt1,1000
gpt_prompt2,1000
gpt_semantic,1000
gpt_writing,1000
human,1000


## Build the Ghostbuster detector pool

Domains:

```text
wp, reuter, essay
```

For each domain we first construct document-ID maps for human, GPT, and Claude, then retain their **intersection**. This matters for Reuters because the public `reuter/gpt` tree contains additional generated files beyond the matched source set. The detector experiment uses the 1000 matched documents per domain.

Expected final counts:

- original GPT/human: 6000
- unseen Claude: 3000
- total: 9000

In [5]:
DOMAINS = ["wp", "reuter", "essay"]

detector_rows = []

# Build each domain from the intersection of document IDs that exist in
# human, GPT, and Claude. This is important for Reuters: the public GPT
# directory contains extra generated files, while the matched human/Claude
# corpus contains 1000 source documents. The paper's detector experiment
# uses the matched 1000-document domain pool, not every .txt file present
# under reuter/gpt.

for domain in DOMAINS:
    roots = {
        generator: GB_DATA / domain / generator
        for generator in ["human", "gpt", "claude"]
    }

    by_generator = {}

    for generator, root in roots.items():
        files = text_files(root, domain)

        mapping = {
            str(document_id(p, root, domain)): p
            for p in files
        }

        assert len(mapping) == len(files), (
            f"Duplicate document IDs in {domain}/{generator}: "
            f"{len(files)} files -> {len(mapping)} IDs"
        )

        by_generator[generator] = mapping
        print(
            domain,
            generator,
            "raw docs:",
            len(mapping),
        )

    common_ids = (
        set(by_generator["human"])
        & set(by_generator["gpt"])
        & set(by_generator["claude"])
    )

    # Stable ordering.
    if domain in {"wp", "essay"}:
        matched_ids = sorted(common_ids, key=int)
    else:
        matched_ids = sorted(common_ids)

    print(
        domain,
        "matched human/gpt/claude docs:",
        len(matched_ids),
    )

    assert len(matched_ids) == 1000, (
        domain,
        len(matched_ids),
    )

    # Original GPT/human pool.
    for generator in ["gpt", "human"]:
        for gid_str in matched_ids:
            p = by_generator[generator][gid_str]
            gid = (
                int(gid_str)
                if domain in {"wp", "essay"}
                else gid_str
            )
            text = read_text(p)

            detector_rows.append({
                "row_id": f"original::{domain}::{generator}::{gid}",
                "file": rel_key(p),
                "source": "original",
                "domain": domain,
                "generator": generator,
                "label": int(generator == "gpt"),
                "group_id": gid,
                "text": text,
                "text_sha256": sha256_text(text),
            })

    # Unseen-Claude evaluation pool.
    for gid_str in matched_ids:
        p = by_generator["claude"][gid_str]
        gid = (
            int(gid_str)
            if domain in {"wp", "essay"}
            else gid_str
        )
        text = read_text(p)

        detector_rows.append({
            "row_id": f"claude_eval::{domain}::claude::{gid}",
            "file": rel_key(p),
            "source": "claude_eval",
            "domain": domain,
            "generator": "claude",
            "label": 1,
            "group_id": gid,
            "text": text,
            "text_sha256": sha256_text(text),
        })

detector = pd.DataFrame(detector_rows)

# Parquet requires a single physical type per column. Detector group IDs are
# numeric for essay/wp and path-like strings for Reuters, so normalize them
# to string before serialization.
detector["group_id"] = detector["group_id"].astype(str)

detector["manifest_index"] = np.arange(len(detector))

assert not detector["row_id"].duplicated().any()

counts = (
    detector
    .groupby(["source", "domain", "generator", "label"])
    .size()
    .rename("n")
    .reset_index()
)

display(counts)

n_original = int(
    (detector["source"] == "original").sum()
)
n_claude = int(
    (detector["source"] == "claude_eval").sum()
)

print("n_original:", n_original)
print("n_claude:", n_claude)
print("total:", len(detector))

assert n_original == 6000, n_original
assert n_claude == 3000, n_claude
assert len(detector) == 9000, len(detector)

wp human raw docs: 1000
wp gpt raw docs: 1000
wp claude raw docs: 1000
wp matched human/gpt/claude docs: 1000
reuter human raw docs: 1000
reuter gpt raw docs: 2000
reuter claude raw docs: 1000
reuter matched human/gpt/claude docs: 1000
essay human raw docs: 1000
essay gpt raw docs: 1000
essay claude raw docs: 1000
essay matched human/gpt/claude docs: 1000


,source,domain,generator,label,n
0,claude_eval,essay,claude,1,1000
1,claude_eval,reuter,claude,1,1000
2,claude_eval,wp,claude,1,1000
3,original,essay,gpt,1,1000
4,original,essay,human,0,1000
5,original,reuter,gpt,1,1000
6,original,reuter,human,0,1000
7,original,wp,gpt,1,1000
8,original,wp,human,0,1000


n_original: 6000
n_claude: 3000
total: 9000


## Save raw pool and manifest

In [6]:
POOL_OUT = RAW / "ghostbuster_detector_pool.parquet"
MANIFEST_OUT = SPLITS / "ghostbuster_detector_manifest.parquet"

detector.to_parquet(
    POOL_OUT,
    index=False,
)

detector.drop(columns=["text"]).to_parquet(
    MANIFEST_OUT,
    index=False,
)

print("saved:", POOL_OUT)
print("saved:", MANIFEST_OUT)
print("shape:", detector.shape)

saved: ../data/raw/ghostbuster_detector_pool.parquet
saved: ../data/splits/ghostbuster_detector_manifest.parquet
shape: (9000, 10)


## Final audit

In [7]:
expected = {
    ("original", "wp", "gpt"): 1000,
    ("original", "wp", "human"): 1000,
    ("original", "reuter", "gpt"): 1000,
    ("original", "reuter", "human"): 1000,
    ("original", "essay", "gpt"): 1000,
    ("original", "essay", "human"): 1000,
    ("claude_eval", "wp", "claude"): 1000,
    ("claude_eval", "reuter", "claude"): 1000,
    ("claude_eval", "essay", "claude"): 1000,
}

actual = (
    detector
    .groupby(["source", "domain", "generator"])
    .size()
    .to_dict()
)

for key, expected_n in expected.items():
    actual_n = actual.get(key)
    assert actual_n == expected_n, (
        f"{key}: expected {expected_n}, got {actual_n}"
    )

# Ensure metadata accidentally found by recursive discovery is absent.
assert not detector["file"].str.contains(
    r"/logprobs/|/prompts/",
    regex=True,
).any()

print("All counts and document-path checks passed.")

All counts and document-path checks passed.


## Next

Use these raw tables with:

- `00a_compute_context_utility.ipynb`
- `00b_compute_recurrence_frontloading.ipynb`

and use `ghostbuster_detector_pool.parquet` with:

- `00d_score_wavedetect.ipynb`